# 4. Validate accuracy options, evaluate once, and save examples
Use **reviewed** COCO annotations from notebook 2. Tune only on validation, freeze the selected recipe, then evaluate test. Inference functions take only RGB images, model checkpoints, and fixed prompts; they cannot access ground-truth masks. Ground truth is used afterward for metrics and figures.

The candidate recipes compare full-image inference, overlapping detail tiles, rotations/flips, prompt variants, and pooling predictions from up to three trained checkpoints. Mask suppression and pixel ownership resolve duplicate instances and conflicting classes. These are hypotheses, not guaranteed improvements: validation chooses the recipe. Prediction files retain instance masks, scores, and provenance.

The default is inert. No trained examples exist until actual GPU training and evaluation finish.


In [ ]:
from pathlib import Path
import json
import numpy as np
from PIL import Image
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataset_notebooks').is_dir() and (p / 'notebooks/reproduction.py').is_file())
sys.path.insert(0, str(ROOT / 'dataset_notebooks'))
from runtime import fresh_run, write_json, read_json, sha256, environment, load_model, SAM3_REVISION


In [ ]:
RUN_VALIDATION = False
RUN_TEST = False
RUN_FULL_IMAGE_EXAMPLES = False
ALLOW_WEIGHT_DOWNLOAD = False
PRETRAINED_CHECKPOINT = None
DATA = ROOT/'experiments/reproductions/sam3-reviewed-export-v1'
TRAINING_RUN = ROOT/'experiments/reproductions/sam3-training-v1'
VALIDATION_RUN_NAME = 'sam3-validation-v1'
SELECTION = ROOT/'experiments/reproductions'/VALIDATION_RUN_NAME/'locked_selection.json'
TEST_RUN_NAME = 'sam3-test-v1'
FULL_IMAGE_PATHS = []  # Optional original RGB images, no masks or boxes supplied.
FULL_IMAGE_RUN_NAME = 'sam3-full-images-v1'


## Spatial bookkeeping

Overlapping tiles cover every pixel, including right/bottom edges. Inverse rotations and flips restore predictions to the original image coordinates.


In [ ]:
def tile_boxes(width,height,tile,overlap=.5):
    if tile is None:return [(0,0,width,height)]
    if tile<=0 or not 0<=overlap<1:raise ValueError('Invalid tiling options.')
    def starts(length):
        last=max(0,length-tile);step=max(1,int(tile*(1-overlap)))
        return sorted(set([*range(0,last+1,step),last]))
    return [(x,y,min(width,x+tile),min(height,y+tile)) for y in starts(height) for x in starts(width)]


def transform_image(array,rotation,flip):
    value=np.rot90(array,rotation)
    if flip:value=np.flip(value,axis=1)
    return np.ascontiguousarray(value)


def restore_mask(mask,rotation,flip):
    if flip:mask=np.flip(mask,axis=1)
    return np.ascontiguousarray(np.rot90(mask,-rotation))


## Duplicate masks and class conflicts

Apply class-aware mask IoU suppression, then assign overlapping visible pixels to the highest-scoring surviving instance. RLE keeps full-image candidate storage compact. Validation selects score and suppression thresholds.


In [ ]:
def suppress_and_resolve(candidates,iou_threshold,score_threshold):
    from pycocotools import mask as mask_api
    selected=[];by_class={1:[],2:[]}
    for candidate in sorted(candidates,key=lambda x:-x['score']):
        if candidate['score']<score_threshold:continue
        if 'segmentation' in candidate:rle=candidate['segmentation']
        else:rle=mask_api.encode(np.asfortranarray(candidate['mask'].astype('uint8')))
        if not mask_api.area(rle):continue
        previous=by_class[candidate['category_id']]
        if previous and (mask_api.iou([rle],previous,[0]*len(previous))>iou_threshold).any():continue
        selected.append((candidate,rle));previous.append(rle)
    if not selected:return []
    occupied=np.zeros(selected[0][1]['size'],bool);result=[]
    for candidate,rle in selected:
        visible=mask_api.decode(rle).astype(bool) & ~occupied
        if not visible.any():continue
        if 'mask' in candidate:result.append({**candidate,'mask':visible})
        else:
            encoded=mask_api.encode(np.asfortranarray(visible.astype('uint8')))
            encoded['counts']=encoded['counts'].decode('ascii')
            result.append({**candidate,'segmentation':encoded})
        occupied|=visible
    return result


## RGB-only inference

Generate proposals using fixed noun phrases, optional detail tiles, and eight rotation/flip views. No ground-truth masks, boxes, or points enter this function.


In [ ]:
def infer_candidates(model,image,recipe):
    import torch
    from pycocotools import mask as mask_api
    from sam3.model.sam3_image_processor import Sam3Processor
    rgb=np.array(image.convert('RGB'));height,width=rgb.shape[:2];candidates=[]
    processor=Sam3Processor(model,confidence_threshold=.01)
    boxes=tile_boxes(width,height,None)
    if recipe['tile']:boxes+=tile_boxes(width,height,recipe['tile'])
    views=[(0,False)] if not recipe['tta'] else [(r,f) for r in range(4) for f in (False,True)]
    with torch.inference_mode(),torch.autocast('cuda',dtype=torch.bfloat16):
        for x0,y0,x1,y1 in boxes:
            crop=rgb[y0:y1,x0:x1]
            for rotation,flip in views:
                state=processor.set_image(Image.fromarray(transform_image(crop,rotation,flip)))
                for category,prompts in recipe['prompts'].items():
                    for prompt in prompts:
                        processor.reset_all_prompts(state)
                        output=processor.set_text_prompt(prompt=prompt,state=state)
                        for mask,score in zip(output['masks'],output['scores']):
                            local=restore_mask(mask.squeeze(0).cpu().numpy().astype(bool),rotation,flip)
                            full=np.zeros((height,width),bool);full[y0:y1,x0:x1]=local
                            rle=mask_api.encode(np.asfortranarray(full.astype('uint8')))
                            rle['counts']=rle['counts'].decode('ascii')
                            candidates.append({'category_id':int(category),'score':float(score),'segmentation':rle})
    return candidates


def encode_predictions(candidates,image_id):
    from pycocotools import mask as mask_api
    output=[]
    for c in candidates:
        rle=c.get('segmentation')
        if rle is None:
            rle=mask_api.encode(np.asfortranarray(c['mask'].astype('uint8')))
            rle['counts']=rle['counts'].decode('ascii')
        output.append({'image_id':image_id,'category_id':c['category_id'],'score':c['score'],'segmentation':rle})
    return output


## Sequential checkpoint ensembles

Run one model at a time and cache RLE proposals on disk. This keeps GPU memory independent of ensemble size and permits threshold tuning without repeating model inference.


In [ ]:
def predict_split(checkpoints,folder,recipe,cache_dir,allow_download=False):
    import torch,gc
    from pycocotools import mask as mask_api
    folder=Path(folder);cache_dir=Path(cache_dir);cache_dir.mkdir(parents=True,exist_ok=False)
    images=read_json(folder/'annotations.json')['images']
    for info in images:
        if sha256(folder/info['file_name'])!=info['image_sha256']:raise ValueError('Evaluation image changed.')
    # One model at a time keeps GPU memory independent of ensemble size.
    for model_index,checkpoint in enumerate(checkpoints):
        model=load_model(checkpoint,allow_download)
        write_json(cache_dir/f'model-{model_index}.json',{'sha256':model._source_checkpoint_sha256,'path':model._source_checkpoint_path})
        for info in images:
            proposals=infer_candidates(model,Image.open(folder/info['file_name']),recipe)
            write_json(cache_dir/f'{model_index}-{info["id"]}.json',encode_predictions(proposals,info['id']))
        del model;gc.collect();torch.cuda.empty_cache()
    result=[]
    for info in images:
        candidates=[]
        for model_index in range(len(checkpoints)):
            for p in read_json(cache_dir/f'{model_index}-{info["id"]}.json'):
                candidates.append(p)
        result+=encode_predictions(suppress_and_resolve(candidates,recipe['nms_iou'],recipe['score']),info['id'])
    return result


## Independent metrics

COCO mask AP/AR evaluate instance separation; union masks additionally provide per-class pixel IoU and Dice. Empty predictions count as missed objects, and absent classes have undefined pixel metrics rather than invented perfect scores.


In [ ]:
def evaluate_predictions(coco_path,predictions):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    from pycocotools import mask as mask_api
    gt=COCO(str(coco_path));known=set(gt.imgs)
    if any(p['image_id'] not in known or p['category_id'] not in (1,2) for p in predictions):raise ValueError('Unknown image/category.')
    if predictions:dt=gt.loadRes(predictions)
    else:
        dt=COCO();dt.dataset={'images':gt.dataset['images'],'categories':gt.dataset['categories'],'annotations':[]};dt.createIndex()
    evaluator=COCOeval(gt,dt,'segm');evaluator.evaluate();evaluator.accumulate();evaluator.summarize()
    metrics={'mask_AP':float(evaluator.stats[0]),'mask_AP50':float(evaluator.stats[1]),'mask_AR100':float(evaluator.stats[8]),'classes':{}}
    for k,category in enumerate(evaluator.params.catIds):
        precision=evaluator.eval['precision'][:,:,k,0,-1];recall=evaluator.eval['recall'][:,k,0,-1]
        totals=np.zeros(3,dtype=np.int64)
        for image_id,info in gt.imgs.items():
            truth=np.zeros((info['height'],info['width']),bool);pred=truth.copy()
            for ann in gt.loadAnns(gt.getAnnIds(imgIds=[image_id],catIds=[category])):truth|=gt.annToMask(ann).astype(bool)
            for ann in predictions:
                if ann['image_id']==image_id and ann['category_id']==category:pred|=mask_api.decode(ann['segmentation']).astype(bool)
            totals+=np.array([(truth&pred).sum(),truth.sum(),pred.sum()],dtype=np.int64)
        intersection,ntruth,npred=map(int,totals);union=ntruth+npred-intersection
        metrics['classes'][str(category)]={'AP':float(precision[precision>=0].mean()) if (precision>=0).any() else None,
            'AR100':float(recall[recall>=0].mean()) if (recall>=0).any() else None,
            'semantic_IoU':intersection/union if union else None,
            'semantic_Dice':2*intersection/(ntruth+npred) if ntruth+npred else None,
            'intersection_pixels':intersection,'truth_pixels':ntruth,'prediction_pixels':npred}
    return metrics


## Reproducible example selection

Save the lowest-ID image from each held-out campaign, rather than choosing visually favorable results. Figures and NPZ masks record the actual selected checkpoints.


In [ ]:
def save_examples(folder,predictions,output,checkpoint_provenance):
    import matplotlib.pyplot as plt
    from pycocotools.coco import COCO
    from pycocotools import mask as mask_api
    folder=Path(folder);output=Path(output);output.mkdir(parents=True,exist_ok=False)
    gt=COCO(str(folder/'annotations.json'));chosen=[]
    for campaign in sorted({i['campaign'] for i in gt.dataset['images']}):
        chosen.append(min((i for i in gt.dataset['images'] if i['campaign']==campaign),key=lambda x:x['id']))
    if len(chosen)<2:chosen=sorted(gt.dataset['images'],key=lambda x:x['id'])[:2]
    for info in chosen[:2]:
        rgb=np.array(Image.open(folder/info['file_name']).convert('RGB'));truth=rgb.astype(float);pred=rgb.astype(float)
        for index,ann in enumerate(gt.loadAnns(gt.getAnnIds(imgIds=[info['id']]))):
            mask=gt.annToMask(ann).astype(bool);color=np.random.default_rng(index+1).integers(30,255,3)
            truth[mask]=.45*truth[mask]+.55*color
        masks=[];classes=[];scores=[]
        fig,axes=plt.subplots(1,3,figsize=(15,5));axes[0].imshow(rgb);axes[1].imshow(truth.astype('uint8'))
        for index,p in enumerate(p for p in predictions if p['image_id']==info['id']):
            mask=mask_api.decode(p['segmentation']).astype(bool);color=np.random.default_rng(index+1).integers(30,255,3)
            pred[mask]=.45*pred[mask]+.55*color;masks.append(mask);classes.append(p['category_id']);scores.append(p['score'])
            yy,xx=np.where(mask)
            axes[2].text(float(xx.mean()),float(yy.mean()),f'{index+1}: {"T" if p["category_id"]==1 else "W"} {p["score"]:.2f}',color='white',fontsize=7)
        axes[2].imshow(pred.astype('uint8'))
        for ax,title in zip(axes,['RGB','Reviewed instances','Trained SAM 3: T=tobacco, W=weed']):ax.set_title(title);ax.axis('off')
        fig.tight_layout();fig.savefig(output/f'{info["source_key"]}.png',dpi=180);plt.show();plt.close(fig)
        np.savez_compressed(output/f'{info["source_key"]}.npz',masks=np.stack(masks) if masks else np.zeros((0,info['height'],info['width']),bool),classes=classes,scores=scores)
    write_json(output/'provenance.json',{'selection':checkpoint_provenance,'image_ids':[i['id'] for i in chosen[:2]],
        'selection_rule':'Lowest image ID per held-out campaign; not selected for visual quality.'})


## Validation candidates

Compare simple inference first, then test tiles, transformations, prompt variants, and ensembles. Only a validation improvement changes the frozen inference recipe.


In [ ]:
def candidate_recipes():
    base={'tile':None,'tta':False,'prompts':{'1':['tobacco plant'],'2':['weed plant']},'nms_iou':.5,'score':.15}
    return [base,{**base,'tile':256},{**base,'tile':384},
            {**base,'tta':True},{**base,'tile':256,'tta':True},
            {**base,'tile':256,'tta':True,'prompts':{'1':['tobacco plant','tobacco'],'2':['weed plant','weeds']}}]


In [ ]:
if RUN_VALIDATION:
    run=fresh_run(ROOT,VALIDATION_RUN_NAME)
    trials=read_json(TRAINING_RUN/'trials.json')
    ranked=sorted(trials,key=lambda t:-t['best_AP'])
    # Resolve relocated checkpoints by their trial directory name, not historical absolute paths.
    from pathlib import PureWindowsPath
    checkpoints=[TRAINING_RUN/PureWindowsPath(t['folder']).name/'best.pt' for t in ranked[:3]]
    if not checkpoints:raise ValueError('No completed trained checkpoints.')
    recipes=candidate_recipes();scores=[]
    baseline=predict_split([PRETRAINED_CHECKPOINT],DATA/'val',recipes[0],run/'baseline-cache',ALLOW_WEIGHT_DOWNLOAD)
    baseline_metrics=evaluate_predictions(DATA/'val/annotations.json',baseline)
    write_json(run/'baseline.json',{'metrics':baseline_metrics,'predictions':baseline})
    for index,recipe in enumerate(recipes):
        predictions=predict_split(checkpoints[:1],DATA/'val',recipe,run/f'recipe-{index}-cache')
        metrics=evaluate_predictions(DATA/'val/annotations.json',predictions)
        scores.append({'recipe':recipe,'checkpoints':[checkpoints[0].relative_to(ROOT).as_posix()],'metrics':metrics,'cache':str(run/f'recipe-{index}-cache')})
        write_json(run/f'recipe-{index}.json',{'predictions':predictions,**scores[-1]})
    best=max(scores,key=lambda x:x['metrics']['mask_AP'])
    for count in range(2,len(checkpoints)+1):
        predictions=predict_split(checkpoints[:count],DATA/'val',best['recipe'],run/f'ensemble-{count}-cache')
        metrics=evaluate_predictions(DATA/'val/annotations.json',predictions)
        scores.append({'recipe':best['recipe'],'checkpoints':[p.relative_to(ROOT).as_posix() for p in checkpoints[:count]],'metrics':metrics,'cache':str(run/f'ensemble-{count}-cache')})
        write_json(run/f'ensemble-{count}.json',{'predictions':predictions,**scores[-1]})
    best=max(scores,key=lambda x:x['metrics']['mask_AP'])
    # Tune score / NMS jointly on a finite validation-only grid.
    paths=[ROOT/p for p in best['checkpoints']]
    for threshold in (.05,.3,.5):
        for nms in (.3,.7):
            recipe={**best['recipe'],'score':threshold,'nms_iou':nms}
            predictions=[]
            for info in read_json(DATA/'val/annotations.json')['images']:
                proposals=[]
                for model_index in range(len(paths)):
                    proposals+=read_json(Path(best['cache'])/f'{model_index}-{info["id"]}.json')
                predictions+=encode_predictions(suppress_and_resolve(proposals,nms,threshold),info['id'])
            scores.append({'recipe':recipe,'checkpoints':best['checkpoints'],
                'metrics':evaluate_predictions(DATA/'val/annotations.json',predictions)})
    selected={k:v for k,v in max(scores,key=lambda x:x['metrics']['mask_AP']).items() if k!='cache'}
    locked={**selected,'checkpoint_hashes':{p:sha256(ROOT/p) for p in selected['checkpoints']},
        'validation_annotations_sha256':sha256(DATA/'val/annotations.json'),
        'test_annotations_sha256':sha256(DATA/'test/annotations.json'),
        'sam3_revision':SAM3_REVISION,'baseline_validation_metrics':baseline_metrics,
        'baseline_checkpoint_sha256':read_json(run/'baseline-cache/model-0.json')['sha256']}
    write_json(run/'comparison.json',scores);write_json(run/'locked_selection.json',locked)
    print('Frozen validation-selected recipe:',locked)
if RUN_TEST:
    locked=read_json(SELECTION)
    if locked['sam3_revision']!=SAM3_REVISION:raise ValueError('Revision changed.')
    for split in ('val','test'):
        key='validation_annotations_sha256' if split=='val' else 'test_annotations_sha256'
        if sha256(DATA/split/'annotations.json')!=locked[key]:raise ValueError('Reviewed annotations changed after selection.')
    for p,digest in locked['checkpoint_hashes'].items():
        if sha256(ROOT/p)!=digest:raise ValueError('Selected checkpoint changed.')
    run=fresh_run(ROOT,TEST_RUN_NAME);write_json(run/'selection.json',locked)
    predictions=predict_split([ROOT/p for p in locked['checkpoints']],DATA/'test',locked['recipe'],run/'trained-cache')
    baseline=predict_split([PRETRAINED_CHECKPOINT],DATA/'test',candidate_recipes()[0],run/'baseline-cache',ALLOW_WEIGHT_DOWNLOAD)
    if read_json(run/'baseline-cache/model-0.json')['sha256']!=locked['baseline_checkpoint_sha256']:
        raise ValueError('Baseline checkpoint changed since validation.')
    write_json(run/'predictions.json',predictions)
    write_json(run/'metrics.json',{'trained':evaluate_predictions(DATA/'test/annotations.json',predictions),
        'pretrained':evaluate_predictions(DATA/'test/annotations.json',baseline)})
    save_examples(DATA/'test',predictions,run/'examples',locked)
    write_json(run/'environment.json',environment())
if RUN_FULL_IMAGE_EXAMPLES:
    import torch,gc
    locked=read_json(SELECTION);run=fresh_run(ROOT,FULL_IMAGE_RUN_NAME)
    for index,path in enumerate(FULL_IMAGE_PATHS):
        image=Image.open(path).convert('RGB');candidates=[]
        for checkpoint in locked['checkpoints']:
            if sha256(ROOT/checkpoint)!=locked['checkpoint_hashes'][checkpoint]:raise ValueError('Checkpoint changed.')
            model=load_model(ROOT/checkpoint,False)
            candidates+=infer_candidates(model,image,locked['recipe']);del model;gc.collect();torch.cuda.empty_cache()
        result=suppress_and_resolve(candidates,locked['recipe']['nms_iou'],locked['recipe']['score'])
        write_json(run/f'image-{index}.json',{'source_sha256':sha256(path),'predictions':encode_predictions(result,index),'selection':locked})
        import matplotlib.pyplot as plt
        rgb=np.array(image);overlay=rgb.astype(float)
        from pycocotools import mask as mask_api
        for k,p in enumerate(result):
            mask=mask_api.decode(p['segmentation']).astype(bool)
            overlay[mask]=.5*overlay[mask]+.5*np.random.default_rng(k).integers(30,255,3)
        fig,axes=plt.subplots(1,2,figsize=(18,6));axes[0].imshow(rgb);axes[1].imshow(overlay.astype('uint8'))
        for ax in axes:ax.axis('off')
        fig.savefig(run/f'image-{index}.png',dpi=180);plt.show();plt.close(fig)
if not any([RUN_VALIDATION,RUN_TEST,RUN_FULL_IMAGE_EXAMPLES]):print('Evaluation disabled; actual trained outputs require the GPU run.')
